In [1]:
import sympy as sp
from IPython.display import display, Math

# for cleaner simplified expressions
def nice(expr):
    return sp.simplify(sp.factor(sp.trigsimp(expr)))

In [2]:
t, r, theta, phi, kappa = sp.symbols('t r theta phi kappa', real=True)
a = sp.Function('a')(t)

coords = [t, r, theta, phi]
dim = 4

### Metric Input

In [3]:
g = sp.diag(
    1,
    -a**2/(1 - kappa*r**2),
    -a**2*r**2,
    -a**2*r**2*sp.sin(theta)**2
)
g_inv = sp.simplify(g.inv())

display(Math(r"g_{\mu\nu} = " + sp.latex(g)))
display(Math(r"g^{\mu\nu} = " + sp.latex(g_inv)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Christoffel Symbols

In [4]:
Gamma = {}

for rho in range(dim):
    for mu in range(dim):
        #Since Gamma^\rho_{mu nu} = Gamma^\rho_{nu mu}, we only need to compute one of them
        #Thus we can reduce computation time by taking only mu <= nu components
        for nu in range(mu, dim):   
            expr = 0
            for lam in range(dim):
                #From Eq. (10)
                expr += g_inv[rho, lam] * (
                    sp.diff(g[nu, lam], coords[mu]) +
                    sp.diff(g[mu, lam], coords[nu]) -
                    sp.diff(g[mu, nu], coords[lam])
                )
            expr = nice(sp.Rational(1, 2) * expr)

            if expr != 0:
                Gamma[(rho, mu, nu)] = expr #Since we only want non 0 symbols
                if mu != nu:
                    Gamma[(rho, nu, mu)] = expr  #Again exploiting mu<->nu symmetry for printing


print("Nonzero Christoffel symbols:")

already_printed = set()

for rho in range(dim):
    for mu in range(dim):
        for nu in range(mu, dim):
            #To avoid redundancy:
            if (rho, mu, nu) in Gamma and (rho, mu, nu) not in already_printed:
                val = Gamma[(rho, mu, nu)]
                if mu == nu:
                    display(Math(
                        r"\Gamma^{%d}_{%d%d} = %s"
                        % (rho, mu, nu, sp.latex(val))
                    ))
                else:
                    display(Math(
                        r"\Gamma^{%d}_{%d%d} = \Gamma^{%d}_{%d%d} = %s"
                        % (rho, mu, nu, rho, nu, mu, sp.latex(val))
                    ))
                    already_printed.add((rho, nu, mu))
                already_printed.add((rho, mu, nu))

Nonzero Christoffel symbols:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Riemann Tensor

In [5]:
Riemann = {}

def Gamma_val(rho, mu, nu):
    return Gamma.get((rho, mu, nu), 0)

for rho in range(dim):
    for sigma in range(dim):
        for mu in range(dim):
            #We can again leverage the symmetries of the Riemann tensor
            for nu in range(mu + 1, dim):   
                expr = sp.diff(Gamma_val(rho, nu, sigma), coords[mu]) \
                     - sp.diff(Gamma_val(rho, mu, sigma), coords[nu])
                #need one more loop for the dummy index
                for lam in range(dim):
                    expr += Gamma_val(rho, mu, lam) * Gamma_val(lam, nu, sigma)
                    expr -= Gamma_val(rho, nu, lam) * Gamma_val(lam, mu, sigma)

                expr = nice(expr)

                if expr != 0:
                    Riemann[(rho, sigma, mu, nu)] = expr
                    Riemann[(rho, sigma, nu, mu)] = -expr  # antisymmetry


print("Independent nonzero Riemann tensor components:")

for rho in range(dim):
    for sigma in range(dim):
        for mu in range(dim):
            for nu in range(mu + 1, dim):
                if (rho, sigma, mu, nu) in Riemann:
                    val = Riemann[(rho, sigma, mu, nu)]
                    display(Math(
                        r"R^{%d}_{%d%d%d} = %s"
                        % (rho, sigma, mu, nu, sp.latex(val))
                    ))

Independent nonzero Riemann tensor components:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Ricci Tensor

In [6]:
Ricci = {}

for mu in range(dim):
    for nu in range(dim):
        expr = 0
        for rho in range(dim):
            expr += Riemann.get((rho, mu, rho, nu), 0)

        expr = nice(expr)
        if expr != 0:
            Ricci[(mu, nu)] = expr

print("Nonzero Ricci tensor components")

for mu in range(dim):
    for nu in range(dim):
        if (mu, nu) in Ricci:
            display(Math(
                r"R_{%d%d} = %s"
                % (mu, nu, sp.latex(Ricci[(mu, nu)]))
            ))

Nonzero Ricci tensor components


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Ricci Scalar

In [7]:
Ricci_scalar = 0
for (mu, nu), val in Ricci.items():
    Ricci_scalar += g_inv[mu, nu] * val

Ricci_scalar = nice(Ricci_scalar)

print("Ricci scalar")
display(Math(r"R = " + sp.latex(Ricci_scalar)))

Ricci scalar


<IPython.core.display.Math object>